# Filter `data/clean/data.csv` → `data/clean/data_filtered.csv`

Applies the fixes found in `data_analysis.ipynb`, in this order:

0. recover area from land dims: for non-condos `area_m2` is land area (≈ width × length on 94% of rows
   with both), yet some rows with real land dims got an imputed area. Use width × length there instead.
1. drop `property_type == "Land"` (too few rows to learn)
2. drop prices below `MIN_PRICE` (likely deposits / rent)
3. drop price-per-m² outliers: |robust z| of log($/m²) within property type > `Z_MAX`
4. dedupe: rows with identical features collapse to one row with the **median** price
5. drop implausible rows (sanity rules): area too small for the type, < 10 m² per bedroom,
   more than `BED_MAX` bedrooms, bathrooms > bedrooms + 2

A tighter outlier filter (|robust z| > 2.5 within type × district) was tried after step 4 and dropped:
on fixed test rows it helped ridge (~-0.6 MdAPE) but not the neural network (RMSE log +0.008),
which already tolerates outliers through its Huber loss. Removing 400 real-but-unusual listings only cost data.

Output keeps the same columns and order as `data.csv`, so the models can read it unchanged
(set `data.path: data/clean/data_filtered.csv` in `configs/neural_network_config.yml`).

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

LAB = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "engines" / "common.py").exists())
SRC = LAB / "data/clean/data.csv"
DST = LAB / "data/clean/data_filtered.csv"
TARGET = "price_usd"

MIN_PRICE = 15_000      # $; rows below are dropped
Z_MAX = 3.5             # robust z threshold on log price per m2, within property type
DROP_TYPES = ["Land"]

# step 5 (after dedupe)
BED_MAX = 12            # more bedrooms than this = hotel / guesthouse / apartment block, not a home
MIN_AREA = {"Condo": 25, "Flat": 25, "Villa": 50, "Twin Villa": 50, "House": 30, "Link House": 30, "Shophouse": 30}

d = pd.read_csv(SRC, encoding="utf-8-sig", low_memory=False)
COLUMNS = d.columns.tolist()
FEATURES = [c for c in COLUMNS if c != TARGET]
print(f"{SRC.relative_to(LAB)}: {len(d):,} rows, {len(COLUMNS)} columns")

data\clean\data.csv: 12,491 rows, 25 columns


## 0. Recover area from land dims

In [2]:
# 0. recover area_m2 = land width x length where the area was imputed but the land dims are real
fixable = ((d["property_type"] != "Condo") & (d["area_m2_missing"] == 1) & (d["land_dims_missing"] == 0)
           & (d["land_width_m"] > 0) & (d["land_length_m"] > 0))
d.loc[fixable, "area_m2"] = (d["land_width_m"] * d["land_length_m"])[fixable].round(2)
d.loc[fixable, "area_m2_missing"] = 0
print(f"area recovered from land dims: {fixable.sum():,} rows; still imputed: {int(d['area_m2_missing'].sum()):,}")

area recovered from land dims: 453 rows; still imputed: 1,962


## 1–3. Drop bad rows

In [3]:
log_ppm2 = np.log(d[TARGET] / d["area_m2"])
med = log_ppm2.groupby(d["property_type"]).transform("median")
mad = (log_ppm2 - med).abs().groupby(d["property_type"]).transform("median")
z = (log_ppm2 - med) / (1.4826 * mad)

reason = pd.Series("", index=d.index)
reason[d["property_type"].isin(DROP_TYPES)] = "property type " + "/".join(DROP_TYPES)
reason[(reason == "") & (d[TARGET] < MIN_PRICE)] = f"price < ${MIN_PRICE:,}"
reason[(reason == "") & (z.abs() > Z_MAX)] = f"price/m2 |z| > {Z_MAX}"

dropped = d[reason != ""].assign(reason=reason[reason != ""], ppm2=(d[TARGET] / d["area_m2"]).round(1), z=z.round(2))
kept = d[reason == ""]
print(reason[reason != ""].value_counts().to_string())
print(f"kept {len(kept):,} of {len(d):,}")

price/m2 |z| > 3.5    166
price < $15,000        38
property type Land     16
kept 12,271 of 12,491


In [4]:
# spot check what goes: the extremes of each reason
dropped.sort_values("z", key=abs, ascending=False).groupby("reason").head(5)[
    ["reason", TARGET, "property_type", "district", "area_m2", "bedrooms", "ppm2", "z"]]

,reason,price_usd,property_type,district,area_m2,bedrooms,ppm2,z
10503,price/m2 |z| > 3.5,3850000.0,Twin Villa,Chbar Ampov,165.00,4.0,23333.3,11.60
9992,price/m2 |z| > 3.5,95000.0,Link House,Mean Chey,2000.00,6.0,47.5,-9.08
9981,price/m2 |z| > 3.5,3200000.0,Twin Villa,Daun Penh,247.00,9.0,12955.5,8.98
10217,price/m2 |z| > 3.5,5000000.0,Twin Villa,Boeng Keng Kang,391.00,5.0,12787.7,8.92
10191,price/m2 |z| > 3.5,12000000.0,Villa,Chroy Changvar,156.25,5.0,76800.0,7.36
2636,"price < $15,000",10200.0,Villa,Battambang,128.00,3.0,79.7,-5.96
2219,"price < $15,000",13500.0,Villa,Kong Pisei,160.00,2.0,84.4,-5.85
578,"price < $15,000",5060.0,Flat,7 Makara,135.00,5.0,37.5,-5.68
951,"price < $15,000",13800.0,Villa,Mean Chey,130.00,4.0,106.2,-5.41
4702,"price < $15,000",10000.0,Condo,Chroy Changvar,50.00,1.0,200.0,-4.67


## 4. Dedupe identical listings

In [5]:
groups = kept.groupby(FEATURES, dropna=False, sort=False)
size = groups[TARGET].transform("size")
spread = groups[TARGET].transform("max") / groups[TARGET].transform("min")
print(f"{(size > 1).sum():,} rows in {(groups.size() > 1).sum():,} duplicate groups; "
      f"max/min price within group p50 {spread[size > 1].median():.2f}, p90 {spread[size > 1].quantile(.9):.2f}")

out = groups[TARGET].median().reset_index()[COLUMNS]
print(f"after dedupe: {len(kept):,} -> {len(out):,} rows")

2,678 rows in 896 duplicate groups; max/min price within group p50 1.39, p90 2.70
after dedupe: 12,271 -> 10,489 rows


## 5. Implausible rows (sanity rules)

In [6]:
real = out["area_m2_missing"] == 0
beds = out["bedrooms_missing"] == 0
baths = out["bathrooms_missing"] == 0

reason5 = pd.Series("", index=out.index)
def drop(mask, why):
    reason5[(reason5 == "") & mask.fillna(False)] = why

drop(real & (out["area_m2"] < out["property_type"].map(MIN_AREA)), "area too small for type")
drop(real & beds & (out["bedrooms"] > 0) & (out["area_m2"] / out["bedrooms"] < 10), "area/bedroom < 10 m2")
drop(beds & (out["bedrooms"] > BED_MAX), f"bedrooms > {BED_MAX}")
drop(beds & baths & (out["bathrooms"] > out["bedrooms"] + 2), "bathrooms > bedrooms + 2")

dropped5 = out[reason5 != ""].assign(reason=reason5[reason5 != ""], ppm2=(out[TARGET] / out["area_m2"]).round(1))
print(reason5[reason5 != ""].value_counts().to_string())
print(f"kept {(reason5 == '').sum():,} of {len(out):,}")

bathrooms > bedrooms + 2    65
area too small for type     44
bedrooms > 12               44
area/bedroom < 10 m2        37
kept 10,299 of 10,489


In [7]:
# spot check what goes: a few rows per reason
dropped5.groupby("reason").head(4)[
    ["reason", TARGET, "property_type", "district", "area_m2", "area_m2_missing", "bedrooms", "bathrooms", "ppm2"]]

,reason,price_usd,property_type,district,area_m2,area_m2_missing,bedrooms,bathrooms,ppm2
86,area too small for type,28150.0,Condo,Sen Sok,23.0,0,1.0,1.0,1223.9
110,area too small for type,59000.0,Condo,Sen Sok,24.0,0,2.0,2.0,2458.3
384,area too small for type,45000.0,House,Dangkao,18.0,0,2.0,3.0,2500.0
745,bathrooms > bedrooms + 2,6500000.0,Villa,Daun Penh,990.0,0,3.0,6.0,6565.7
1090,area too small for type,118000.0,Villa,Sen Sok,40.0,0,3.0,4.0,2950.0
1621,area/bedroom < 10 m2,430000.0,Shophouse,Chamkar Mon,40.0,0,5.0,5.0,10750.0
1859,area/bedroom < 10 m2,90000.0,House,Dangkao,49.2,0,5.0,5.0,1829.3
2668,bedrooms > 12,3000000.0,Villa,Tuol Kouk,560.0,0,16.0,8.0,5357.1
2706,bedrooms > 12,3600000.0,Villa,Boeng Keng Kang,567.0,1,14.0,8.0,6349.2
4926,bathrooms > bedrooms + 2,3000000.0,Villa,Chbar Ampov,1890.0,0,6.0,12.0,1587.3


In [8]:
out = out[reason5 == ""].reset_index(drop=True)
print(f"after step 5: {len(out):,} rows")

after step 5: 10,299 rows


## Checks and save

In [9]:
assert out.columns.tolist() == COLUMNS
assert not out.duplicated(FEATURES).any()
assert out.isna().sum().sum() == 0
assert (out[TARGET] >= MIN_PRICE).all() and not out["property_type"].isin(DROP_TYPES).any()
# keep dtypes identical to the source (groupby can upcast)
out = out.astype(d.dtypes.to_dict())

before = d["property_type"].value_counts().rename("before")
after = out["property_type"].value_counts().rename("after")
cmp = pd.concat([before, after], axis=1).fillna(0).astype(int)
cmp["removed"] = (1 - cmp["after"] / cmp["before"]).map("{:.1%}".format)
display(cmp)
print(f"price median ${d[TARGET].median():,.0f} -> ${out[TARGET].median():,.0f}; "
      f"max ${d[TARGET].max():,.0f} -> ${out[TARGET].max():,.0f}")

,before,after,removed
property_type,,,
Condo,5369,3864,28.0%
Villa,2279,2049,10.1%
Flat,1510,1379,8.7%
House,1313,1237,5.8%
Shophouse,1190,1055,11.3%
Twin Villa,468,402,14.1%
Link House,346,313,9.5%
Land,16,0,100.0%


price median $155,000 -> $168,500; max $15,000,000 -> $15,000,000


In [10]:
out.to_csv(DST, index=False, encoding="utf-8-sig")
print(f"wrote {DST.relative_to(LAB)}: {len(out):,} rows ({1 - len(out) / len(d):.1%} removed from {len(d):,})")

wrote data\clean\data_filtered.csv: 10,299 rows (17.5% removed from 12,491)
